# 🎯 Hull Tactical Market Prediction - Simplified Pipeline

## ⚠️ IMPORTANT: This notebook uses pre-built optimization scripts!

**This is a SIMPLIFIED version - just 3 script calls!**

## 📌 Setup Instructions

### Step 1: Create ZIP file (Already done! ✅)

```bash
bash create_kaggle_zip.sh
```

This creates `prediction_market_modules.zip` containing:
- `src/` - All Python modules (data, features, models, etc.)
- `scripts/` - Optimization scripts
- `conf/params.yaml` - Configuration file

### Step 2: Upload Dataset to Kaggle

1. **Go to**: https://www.kaggle.com/datasets
2. **Click**: "New Dataset"
3. **Upload**: `prediction_market_modules.zip`
4. **Name it**: 원하는 이름 (예: `prediction-market-modules`)
5. **Click**: "Create"

### Step 3: Configure Dataset Name

**⚠️ IMPORTANT**: Cell 2에서 데이터셋 이름을 수정하세요!

```python
DATASET_NAME = "your-dataset-name"  # 변경 필요!
```

### Step 4: Run Notebook

1. **Click**: "Run All"
2. **Wait**: ~20-30 minutes (3 scripts running sequentially)
3. **Download**: `submissions/submission.parquet`
4. **Submit**: Upload to competition

## 🚀 What This Notebook Does:

### Script 1: optimize_return_model.py (~10 min)
- Load and preprocess train.csv
- Feature engineering (600+ features)
- Feature selection (weighted ensemble)
- Train LightGBM return model (5-fold CV)
- Save model to artifacts/

### Script 2: optimize_risk_model.py (~10 min)
- Load and preprocess train.csv
- Create risk labels (future volatility)
- Feature engineering + selection
- Train LightGBM risk model (5-fold CV)
- Save model to artifacts/

### Script 3: optimize_position_strategy.py (~10 min)
- Load saved return & risk models
- Optimize position mapping (Sharpe Scaling vs Quantile Binning)
- Load test.csv
- Generate predictions (r_hat, sigma_hat)
- Map to allocations (0.0 to 2.0)
- **Save submission.parquet** ✅

## 🎯 Advantages:

- ✅ **Much Cleaner**: 3 script calls instead of 200+ lines
- ✅ **Modular**: Each component independently debuggable
- ✅ **Reusable**: Same scripts work locally and on Kaggle
- ✅ **Maintainable**: Update scripts without touching notebook
- ✅ **Testable**: Can test each script locally before uploading

## 1️⃣ Setup: Configure Dataset Path

In [ ]:
import os
import sys
from pathlib import Path
import yaml

# ========== CONFIGURATION: 데이터셋 이름 (여기만 수정하세요!) ==========
DATASET_NAME = "prediction-market-modules"
# ====================================================================

print("="*80)
print("SETTING UP PATHS & GPU CONFIGURATION")
print("="*80)

# Add dataset to Python path
DATASET_PATH = Path(f"/kaggle/input/{DATASET_NAME}")

if DATASET_PATH.exists():
    sys.path.insert(0, str(DATASET_PATH))
    print(f"✓ Dataset found: {DATASET_PATH}")
    
    # Check for required directories
    required_dirs = ['src', 'scripts', 'conf']
    for dir_name in required_dirs:
        dir_path = DATASET_PATH / dir_name
        if dir_path.exists():
            print(f"  ✓ {dir_name}/ exists")
        else:
            print(f"  ❌ {dir_name}/ NOT FOUND!")
    
    # Set working directory (for scripts to save results)
    os.chdir("/kaggle/working")
    print(f"\n✓ Working directory: {os.getcwd()}")
    
else:
    print(f"❌ Dataset not found: {DATASET_PATH}")
    print(f"\nAvailable datasets:")
    input_dir = Path("/kaggle/input/")
    if input_dir.exists():
        for item in input_dir.iterdir():
            print(f"  📁 {item.name}")
    raise FileNotFoundError(f"Dataset '{DATASET_NAME}' not found!")

# ========== GPU CONFIGURATION ==========
print(f"\n{'='*80}")
print("GPU CONFIGURATION")
print("="*80)

def detect_and_configure_gpu():
    """Detect environment and configure GPU for LightGBM."""
    
    # Check if running on Kaggle (CUDA GPU available)
    if Path("/kaggle").exists():
        try:
            import torch
            if torch.cuda.is_available():
                device = "gpu"
                gpu_platform = "cuda"
                n_gpus = torch.cuda.device_count()
                gpu_name = torch.cuda.get_device_name(0) if n_gpus > 0 else "Unknown"
                print(f"✓ Kaggle Environment Detected")
                print(f"  Device: {device} ({gpu_platform})")
                print(f"  GPU: {gpu_name}")
                print(f"  GPU Count: {n_gpus}")
                return device, gpu_platform
        except ImportError:
            print("⚠️  PyTorch not available, falling back to CPU")
    
    # Check for Apple Silicon (MPS)
    try:
        import torch
        if torch.backends.mps.is_available():
            device = "mps"
            gpu_platform = "apple_silicon"
            print(f"✓ Apple Silicon Detected")
            print(f"  Device: {device} (Metal Performance Shaders)")
            return device, gpu_platform
    except (ImportError, AttributeError):
        pass
    
    # Fallback to CPU
    device = "cpu"
    gpu_platform = "none"
    print(f"ℹ️  Using CPU")
    print(f"  Device: {device}")
    return device, gpu_platform

device, gpu_platform = detect_and_configure_gpu()

# Update config file with GPU settings
config_path = DATASET_PATH / "conf" / "params.yaml"
if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Update return model config
    if 'model_return' in config and 'lightgbm' in config['model_return']:
        config['model_return']['lightgbm']['device'] = device
        if device == "gpu":
            config['model_return']['lightgbm']['gpu_platform_id'] = 0
            config['model_return']['lightgbm']['gpu_device_id'] = 0
    
    # Update risk model config
    if 'risk' in config and 'lightgbm' in config['risk']:
        if 'fixed_params' in config['risk']['lightgbm']:
            config['risk']['lightgbm']['fixed_params']['device'] = device
            if device == "gpu":
                config['risk']['lightgbm']['fixed_params']['gpu_platform_id'] = 0
                config['risk']['lightgbm']['fixed_params']['gpu_device_id'] = 0
    
    # Save updated config to working directory
    working_config_dir = Path("/kaggle/working/conf")
    working_config_dir.mkdir(exist_ok=True)
    working_config_path = working_config_dir / "params.yaml"
    
    with open(working_config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"\n✓ Config updated with GPU settings")
    print(f"  Saved to: {working_config_path}")
    print(f"  Device: {device}")
else:
    print(f"\n⚠️  Config file not found: {config_path}")

print("\n✅ Path setup & GPU configuration complete!")

## 2️⃣ Script 1: Train Return Model (~10 min)

In [ ]:
print("="*80)
print("STEP 1/3: TRAINING RETURN MODEL")
print("="*80)

# Copy dataset files to working directory for script access
import shutil

# Copy data files
data_dir = Path("/kaggle/working/data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

train_src = Path("/kaggle/input/hull-tactical-market-prediction/train.csv")
test_src = Path("/kaggle/input/hull-tactical-market-prediction/test.csv")

if train_src.exists():
    shutil.copy(train_src, data_dir / "train.csv")
    print(f"✓ Copied train.csv to {data_dir}")

if test_src.exists():
    shutil.copy(test_src, data_dir / "test.csv")
    print(f"✓ Copied test.csv to {data_dir}")

# Run optimize_return_model.py
script_path = DATASET_PATH / "scripts" / "optimize_return_model.py"

if script_path.exists():
    print(f"\nRunning: {script_path}")
    print(f"Using GPU config: {device}\n")
    %run {script_path}
    print("\n✅ Return model training complete!")
else:
    raise FileNotFoundError(f"Script not found: {script_path}")

print("\n📊 Results:")
print("  - Trained model saved to: artifacts/models_optimized/")
print("  - Feature list saved to: results/feature_selection/selected_features_optimized.csv")
print("  - OOF predictions saved for position optimization")

## 3️⃣ Script 2: Train Risk Model (~10 min)

In [ ]:
print("="*80)
print("STEP 2/3: TRAINING RISK MODEL")
print("="*80)

# Run optimize_risk_model.py
script_path = DATASET_PATH / "scripts" / "optimize_risk_model.py"

if script_path.exists():
    print(f"Running: {script_path}")
    print(f"Using GPU config: {device}\n")
    %run {script_path}
    print("\n✅ Risk model training complete!")
else:
    raise FileNotFoundError(f"Script not found: {script_path}")

print("\n📊 Results:")
print("  - Trained model saved to: artifacts/models_risk_optimized/")
print("  - Feature list saved to: results/feature_selection/selected_features_risk_optimized.csv")
print("  - OOF risk predictions saved for position optimization")

## 4️⃣ Script 3: Optimize Position Strategy + Generate Submission (~10 min)

In [ ]:
print("="*80)
print("STEP 3/3: OPTIMIZING POSITION STRATEGY & GENERATING SUBMISSION")
print("="*80)

# Run optimize_position_strategy.py
script_path = DATASET_PATH / "scripts" / "optimize_position_strategy.py"

if script_path.exists():
    print(f"Running: {script_path}")
    print(f"Using GPU config: {device}\n")
    %run {script_path}
    print("\n✅ Position optimization & submission generation complete!")
else:
    raise FileNotFoundError(f"Script not found: {script_path}")

print("\n📊 Results:")
print("  - Best strategy saved to: artifacts/best_position_strategy.json")
print("  - Submission file: submissions/submission.parquet")

# Verify submission file
submission_path = Path("submissions/submission.parquet")
if submission_path.exists():
    import pandas as pd
    submission = pd.read_parquet(submission_path)
    print(f"\n✅ Submission verified:")
    print(f"  - Rows: {len(submission)}")
    print(f"  - Columns: {list(submission.columns)}")
    print(f"  - Allocation range: [{submission['allocation'].min():.4f}, {submission['allocation'].max():.4f}]")
    print(f"  - Mean allocation: {submission['allocation'].mean():.4f}")
    
    print(f"\n{'='*80}")
    print("GPU USAGE SUMMARY")
    print("="*80)
    print(f"  Environment: {'Kaggle' if Path('/kaggle').exists() else 'Local'}")
    print(f"  Device Used: {device}")
    print(f"  Platform: {gpu_platform}")
    
    print(f"\n📥 Download 'submissions/submission.parquet' and submit to competition!")
else:
    print(f"\n❌ Submission file not found: {submission_path}")

## 📊 Pipeline Summary

### Complete Workflow:

```
Cell 1: Setup paths
   ↓
Cell 2: Run optimize_return_model.py (~10 min)
   ↓ Trains return model + saves to artifacts/
   ↓
Cell 3: Run optimize_risk_model.py (~10 min)
   ↓ Trains risk model + saves to artifacts/
   ↓
Cell 4: Run optimize_position_strategy.py (~10 min)
   ↓ Optimizes position mapping
   ↓ Loads test.csv
   ↓ Generates predictions
   ↓ Creates submission.parquet ✅
```

### Total Time: ~30 minutes

### Key Advantages:

| Aspect | Old Notebook | New Notebook |
|--------|--------------|--------------|
| **Lines of Code** | ~600 lines | ~100 lines |
| **Maintainability** | Hard (inline code) | Easy (modular scripts) |
| **Debugging** | Difficult | Simple (test scripts locally) |
| **Reusability** | Low | High (same scripts everywhere) |
| **Readability** | Complex | Clear (3 simple steps) |

### What Each Script Does:

**1. optimize_return_model.py:**
- Load train.csv
- Preprocess (winsorize, normalize, scale)
- Feature engineering (600+ features)
- Feature selection (weighted ensemble → ~150 features)
- Train LightGBM (5-fold CV)
- Save model to `artifacts/models_optimized/`

**2. optimize_risk_model.py:**
- Load train.csv
- Create risk labels (future volatility)
- Preprocess + feature engineering
- Feature selection (weighted ensemble → ~150 features)
- Train LightGBM (5-fold CV)
- Save model to `artifacts/models_risk_optimized/`

**3. optimize_position_strategy.py:**
- Load saved return & risk models
- Generate OOF predictions for optimization
- Test 2 strategies: Sharpe Scaling vs Quantile Binning
- Optimize parameters with Optuna
- Select best strategy
- **Load test.csv** ✨
- **Generate return & risk predictions** ✨
- **Map to allocations (0.0 to 2.0)** ✨
- **Save submission.parquet** ✨

### 🎯 Ready for Submission!

1. ✅ Upload `prediction_market_modules.zip` to Kaggle Datasets
2. ✅ Add dataset to this notebook
3. ✅ Update `DATASET_NAME` in Cell 1
4. ✅ Click "Run All"
5. ✅ Wait ~30 minutes
6. ✅ Download `submissions/submission.parquet`
7. ✅ Submit to competition!

**Good luck! 🚀**